In [20]:
import jax
import jax.numpy as jnp
from flax import linen as nn
import optax  # The standard optimization library for JAX

class Regressor(nn.Module):
    @nn.compact
    def __call__(self, x):
        x = nn.Dense(features=16)(x)
        x = nn.relu(x)
        x = nn.Dense(features=4)(x)  # 4 Output neurons
        return x

def mse_loss(params, x, y, model):
    # Forward pass happens inside the loss function
    preds = model.apply({'params': params}, x)
    return jnp.mean((preds - y) ** 2)

# 1. Setup Data
key = jax.random.PRNGKey(0)
x_dummy = jax.random.normal(key, (10, 8)) # 10 samples, 8 features
y_dummy = jax.random.normal(key, (10, 4)) # 10 samples, 4 targets

# 2. Initialize Model and Optimizer
model = Regressor()
variables = model.init(key, x_dummy)
params = variables['params']
optimizer = optax.sgd(learning_rate=0.01)
opt_state = optimizer.init(params)

# 3. Training Step (The Backward Pass)
# @jax.jit # Compiles the function for speed
def train_step(params, opt_state, x, y):
    # Calculate gradients
    loss_val, grads = jax.value_and_grad(mse_loss)(params, x, y, model)
    print(grads["Dense_0"]["kernel"].shape)
    
    # Update parameters
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    
    return params, opt_state, loss_val

# --- Execution ---
# Forward Pass Only
prediction = model.apply({'params': params}, x_dummy)

# Backward Pass (One Step)
params, opt_state, loss = train_step(params, opt_state, x_dummy, y_dummy)

print(f"Initial Loss: {loss}")

(8, 16)
Initial Loss: 1.5096492767333984


In [11]:
params.keys()

dict_keys(['Dense_0', 'Dense_1'])